# Federated Learning + Homomorphic Encryption + XGBoost — Demo

This notebook is a short, interactive walkthrough of the pipeline implemented in this repository:

`Dataset Upload -> Federated Learning (3 organizations) -> Homomorphic Encryption of updates -> Secure Aggregation -> Global XGBoost Model`

It runs a **small** number of communication rounds so it finishes quickly; for the full run use `python train.py`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import config

# Keep this demo fast - override just for this notebook run
config.COMMUNICATION_ROUNDS = 3
config.LOCAL_EPOCHS = 10

from federated.trainer import FederatedTrainer

print("Clients:", config.NUM_CLIENTS)
print("Rounds:", config.COMMUNICATION_ROUNDS)
print("Local epochs per round:", config.LOCAL_EPOCHS)


## 1. Load data and set up clients + server

In [ ]:
trainer = FederatedTrainer(seed=config.RANDOM_SEED)
trainer.setup()

print("Classes:", trainer.global_classes)
for cid, client in trainer.clients.items():
    print(f"Client {cid}: train={client.dataset.n_train_samples}, val={len(client.dataset.y_val)}")


## 2. Run federated training

Each round: local XGBoost training on every client -> local evaluation -> feature-importance +
performance update vector -> **CKKS homomorphic encryption** of that vector -> **encrypted
weighted aggregation** on the server -> decryption + validation against the plaintext expected
value -> global weighted-ensemble model rebuilt -> evaluated on the shared held-out test set.


In [ ]:
history = trainer.run(rounds=config.COMMUNICATION_ROUNDS, local_epochs=config.LOCAL_EPOCHS)

for r in history:
    print(f"Round {r.round_num}: global {r.global_metrics.summary()} | "
          f"HE validation {'PASSED' if r.validation_passed else 'FAILED'} "
          f"(max_abs_error={r.validation_max_error:.6f})")


## 3. Per-client local performance (last round)

In [ ]:
last_round = history[-1]
for cid, update in last_round.local_updates.items():
    print(f"Client {cid}: {update.metrics.summary()}  (n_samples={update.n_samples})")


## 4. Blockchain-ready metadata (per client, per round)

In [ ]:
for record in last_round.blockchain_records:
    print(record)


## 5. Visualizations

In [ ]:
from utils.visualization import (
    plot_round_metric, plot_client_comparison, plot_confusion_matrix, plot_roc_curve,
)

rounds = [r.round_num for r in history]
client_ids = sorted(trainer.clients.keys())

global_acc = [r.global_metrics.accuracy for r in history]
global_f1 = [r.global_metrics.f1_macro for r in history]
global_loss = [r.global_metrics.loss for r in history]

client_acc = {cid: [r.local_updates[cid].metrics.accuracy for r in history] for cid in client_ids}
client_f1 = {cid: [r.local_updates[cid].metrics.f1_macro for r in history] for cid in client_ids}

path1 = plot_round_metric(rounds, global_acc, "Accuracy", "notebook_round_vs_accuracy.png", client_acc)
path2 = plot_round_metric(rounds, global_f1, "F1 Score (macro)", "notebook_round_vs_f1.png", client_f1)
path3 = plot_round_metric(rounds, global_loss, "Loss (log-loss)", "notebook_round_vs_loss.png")
print(path1, path2, path3, sep="\n")


In [ ]:
from IPython.display import Image, display
display(Image(filename=path1))
display(Image(filename=path2))
display(Image(filename=path3))


In [ ]:
path4 = plot_client_comparison(
    client_ids=last_round.participating_clients,
    metric_values=[last_round.local_updates[c].metrics.f1_macro for c in last_round.participating_clients],
    metric_name="F1 Score (macro)",
    round_num=last_round.round_num,
    filename="notebook_client_comparison.png",
)
display(Image(filename=path4))


In [ ]:
import numpy as np

path5 = plot_confusion_matrix(
    np.array(last_round.global_metrics.confusion_matrix),
    classes=trainer.global_classes,
    filename="notebook_confusion_matrix.png",
    title=f"Global Model Confusion Matrix (Round {last_round.round_num})",
)
display(Image(filename=path5))


In [ ]:
X_test, y_test, _ = trainer.partitioner.load_global_test()

path6 = plot_roc_curve(
    y_test, last_round.global_proba, classes=trainer.global_classes,
    filename="notebook_roc_curve.png",
)
display(Image(filename=path6))


## Next steps

* Run the full pipeline: `python train.py --rounds 10 --local-epochs 20`
* Evaluate a saved model later: `python evaluate.py`
* All logs are under `logs/`, all metrics + blockchain metadata under `outputs/metadata/round_history.json`.
